# 📊 Notebook 01 — Data Exploration
**Step 4** | `notebooks/01_exploration.ipynb`

Goals:
- Load data into the notebook
- Check shape, data types, and missing values
- Visualise distributions and correlations
- Identify data quality problems

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Load Data

In [ ]:
data = {
    'Order_ID':              [0, 1, 2, 3, 4, 5, 6],
    'Distance_km':           [5, 22, 7.93, 16.42, 19.52, 17.44, 19.03],
    'Weather':               ['Clear','Rainy','Windy','Clear','Foggy','Rainy','Clear'],
    'Traffic_Level':         ['Low','Medium','Low','Medium','Low','Medium','Low'],
    'Time_of_Day':           ['Morning','Evening','Afternoon','Evening','Night','Afternoon','Morning'],
    'Vehicle_Type':          ['Bike','Scooter','Scooter','Bike','Scooter','Scooter','Bike'],
    'Preparation_Time_min':  [10, 25, 12, 20, 28, 5, 16],
    'Courier_Experience_yrs':[3.0, 1.5, 1.0, 2.0, 1.0, 1.0, 5.0],
    'Delivery_Time_min':     [25, 55, 43, 84, 59, 36, 68],
}

df = pd.DataFrame(data)
print('Shape:', df.shape)
df.head()

## 2. Data Types, Nulls, Duplicates

In [ ]:
print('=== dtypes ===')
print(df.dtypes)
print('\n=== Missing values ===')
print(df.isnull().sum())
print('\n=== Duplicates:', df.duplicated().sum(), '===')

In [ ]:
df.describe()

## 3. Categorical Value Counts

In [ ]:
for col in df.select_dtypes(include='object').columns:
    print(f'\n--- {col} ---')
    print(df[col].value_counts())

## 4. Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Delivery_Time_min'], bins=8, color='steelblue', edgecolor='white')
axes[0].set_title('Delivery Time Distribution')
axes[0].set_xlabel('Delivery Time (min)')
axes[0].set_ylabel('Count')

axes[1].boxplot(df['Delivery_Time_min'], patch_artist=True,
                boxprops=dict(facecolor='steelblue', color='navy'))
axes[1].set_title('Delivery Time Box Plot')
axes[1].set_ylabel('Delivery Time (min)')

plt.suptitle('Target Variable — Delivery_Time_min', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Distance vs Delivery Time by Traffic

In [ ]:
colors = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
plt.figure(figsize=(8, 5))
for tl, grp in df.groupby('Traffic_Level'):
    plt.scatter(grp['Distance_km'], grp['Delivery_Time_min'],
                label=tl, color=colors[tl], s=90, edgecolors='white')
plt.title('Distance vs Delivery Time (by Traffic Level)')
plt.xlabel('Distance (km)')
plt.ylabel('Delivery Time (min)')
plt.legend(title='Traffic Level')
plt.tight_layout()
plt.show()

## 6. Average Delivery Time by Categorical Features

In [ ]:
cat_features = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, col in zip(axes.flatten(), cat_features):
    avg = df.groupby(col)['Delivery_Time_min'].mean().sort_values(ascending=False)
    ax.bar(avg.index, avg.values, color='#9b59b6', edgecolor='white')
    ax.set_title(f'Avg Delivery Time by {col}')
    ax.set_ylabel('Avg Delivery Time (min)')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Categorical Features vs Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Correlation Heatmap (Numeric Features)

In [ ]:
num_df = df.select_dtypes(include=np.number).drop(columns=['Order_ID'])
plt.figure(figsize=(8, 5))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 8. Courier Experience vs Delivery Time

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(df['Courier_Experience_yrs'], df['Delivery_Time_min'],
            color='#e67e22', s=80, edgecolors='white')
for _, row in df.iterrows():
    plt.annotate(f"  {row['Vehicle_Type']}",
                 (row['Courier_Experience_yrs'], row['Delivery_Time_min']),
                 fontsize=8, color='grey')
plt.title('Courier Experience vs Delivery Time')
plt.xlabel('Experience (yrs)')
plt.ylabel('Delivery Time (min)')
plt.tight_layout()
plt.show()

## 9. Data Quality Summary

In [ ]:
issues = []
if df.isnull().any().any():  issues.append('❌ Missing values found')
else:                         issues.append('✅ No missing values')
if df.duplicated().any():    issues.append('❌ Duplicate rows found')
else:                         issues.append('✅ No duplicates')
if (df['Distance_km'] <= 0).any(): issues.append('❌ Non-positive Distance_km')
else:                              issues.append('✅ Distance_km looks valid')
if (df['Delivery_Time_min'] <= 0).any(): issues.append('❌ Non-positive target')
else:                                     issues.append('✅ Target variable looks valid')

print('\n'.join(issues))